# Module 7.4: Super-Resolution Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_07_RL_For_Image_Enhancement/04_Super_Resolution_Agent/super_resolution_agent.ipynb)

---

## Learning Objectives

1. **Formulate** image super-resolution as a patch-level decision problem
2. **Implement** multiple upscaling strategies (bicubic, edge-guided, learned CNN upscaler)
3. **Build** an RL agent that selects the best upscaling method per image region
4. **Train** on small synthetic images and measure PSNR against ground truth
5. **Visualise** region-level strategy maps and before/after upscaling results

---

## 1 — Mathematical Foundation

### 1.1 Super-Resolution Problem

Given a low-resolution (LR) image $I_{\text{LR}} \in \mathbb{R}^{h \times w}$, reconstruct a high-resolution (HR) image $I_{\text{HR}} \in \mathbb{R}^{sh \times sw}$ where $s$ is the scale factor. The degradation model:

$$I_{\text{LR}} = (I_{\text{HR}} * k) \downarrow_s + n$$

where $k$ is a blur kernel, $\downarrow_s$ is downsampling by factor $s$, and $n$ is noise.

### 1.2 Upscaling Methods

**Bicubic interpolation:** Uses a $4 \times 4$ neighbourhood with the cubic kernel:

$$W(x) = \begin{cases} (a+2)|x|^3 - (a+3)|x|^2 + 1 & |x| \leq 1 \\ a|x|^3 - 5a|x|^2 + 8a|x| - 4a & 1 < |x| < 2 \\ 0 & \text{otherwise} \end{cases}$$

with $a = -0.5$ (Catmull-Rom). Fast but produces blurry edges.

**Edge-guided interpolation:** Detects edge orientation using the gradient structure tensor:

$$\mathbf{T} = \begin{bmatrix} \sum G_x^2 & \sum G_x G_y \\ \sum G_x G_y & \sum G_y^2 \end{bmatrix}$$

Interpolation is then performed along the dominant edge direction (eigenvector of $\mathbf{T}$ with smallest eigenvalue).

**Learned CNN upscaler:** A small SRCNN-style network with sub-pixel convolution:

$$I_{\text{SR}} = f_{\text{CNN}}(I_{\text{LR}}; \theta), \quad \text{using PixelShuffle}(r)$$

### 1.3 Patch-Level MDP

We divide the LR image into $N$ non-overlapping patches and process each sequentially:

| Component | Definition |
|-----------|------------|
| **State** $s_i$ | Patch feature vector: local statistics + edge strength + texture energy |
| **Action** $a_i$ | Upscaling method for patch $i$ from $\{\text{bicubic}, \text{edge-guided}, \text{learned}\}$ |
| **Reward** | $r_i = \text{PSNR}(\text{patch}_i^{\text{SR}}, \text{patch}_i^{\text{HR}})$ |
| **Episode** | One full image: all $N$ patches processed |

### 1.4 Patch Feature Vector

For each LR patch $P$, we compute:

$$\mathbf{f}(P) = \left[\mu(P),\; \sigma(P),\; E_{\text{edge}},\; E_{\text{texture}},\; \text{flatness}\right]$$

where $E_{\text{edge}} = \|\nabla P\|_1$ (gradient energy), $E_{\text{texture}} = \sigma(|P * L|)$ (Laplacian variance), and flatness $= 1 - \frac{\sigma(P)}{\max(P) - \min(P) + \epsilon}$.

## 2 — Environment Setup

## Deep Derivation: Super-Resolution Theory and Upsampling Mathematics

### Step 1: The Degradation Model
$$I_{LR} = (I_{HR} * k) \downarrow_s + n$$

where $k$ is blur kernel, $\downarrow_s$ is downsampling by factor $s$, $n$ is noise.

**Matrix form:** $\mathbf{y} = \mathbf{D}\mathbf{H}\mathbf{x} + \mathbf{n}$

where $\mathbf{D}$ = downsampling matrix, $\mathbf{H}$ = blur matrix, $\mathbf{x}$ = HR image.

### Step 2: Why SR is Ill-Posed (Proof)
For upscaling factor $s$, we have $\frac{H \times W}{s^2}$ known pixels and $H \times W$ unknowns.

**Proof (underdetermined system):**

$\mathbf{y} \in \mathbb{R}^{n/s^2}$, $\mathbf{x} \in \mathbb{R}^n$. The system $\mathbf{y} = \mathbf{A}\mathbf{x}$ has $n/s^2$ equations and $n$ unknowns.

Null space dimension: $\dim(\text{null}(\mathbf{A})) = n - n/s^2 = n(1 - 1/s^2)$

For $s=4$: null space has dimension $\frac{15}{16}n$ — infinitely many solutions! $\blacksquare$

### Step 3: Interpolation Methods — Mathematical Basis

**Nearest Neighbor:** $I_{HR}(x,y) = I_{LR}(\lfloor x/s \rfloor, \lfloor y/s \rfloor)$ — piecewise constant.

**Bilinear Interpolation:**
$$I(x,y) = (1-\alpha)(1-\beta)I_{00} + \alpha(1-\beta)I_{10} + (1-\alpha)\beta I_{01} + \alpha\beta I_{11}$$

where $\alpha = x - \lfloor x \rfloor$, $\beta = y - \lfloor y \rfloor$.

**Proof bilinear is exact for linear functions:**

Let $f(x,y) = ax + by + c$. Then bilinear interpolation of corner values $f_{00}, f_{10}, f_{01}, f_{11}$:
$$(1-\alpha)(1-\beta)(c) + \alpha(1-\beta)(a+c) + (1-\alpha)\beta(b+c) + \alpha\beta(a+b+c) = a\alpha + b\beta + c = f(\alpha,\beta) \quad \blacksquare$$

**Bicubic:** Uses 4×4 neighborhood with cubic polynomial:
$$h(t) = \begin{cases} (a+2)|t|^3 - (a+3)|t|^2 + 1 & |t| \leq 1 \\ a|t|^3 - 5a|t|^2 + 8a|t| - 4a & 1 < |t| < 2 \\ 0 & |t| \geq 2 \end{cases}$$

### Step 4: Sub-Pixel Convolution (Pixel Shuffle)
Reshape $C \cdot s^2$ channels into $s \times s$ spatial resolution:
$$I_{HR}(s \cdot i + \delta_y, s \cdot j + \delta_x) = F_{(\delta_y \cdot s + \delta_x)}(i, j)$$

for $\delta_x, \delta_y \in \{0, 1, \ldots, s-1\}$.

**Proof this is lossless** (bijective mapping):

Each output pixel $(s \cdot i + \delta_y, s \cdot j + \delta_x)$ maps to exactly one channel $(\delta_y \cdot s + \delta_x)$ at position $(i,j)$. Total pixels: $s^2 \cdot \frac{H}{s} \cdot \frac{W}{s} = H \times W$. $\blacksquare$

### Step 5: Perceptual Loss for SR
Instead of pixel-wise MSE, use feature-space loss:
$$L_{\text{perceptual}} = \frac{1}{C_l H_l W_l}\|\phi_l(I_{HR}) - \phi_l(\hat{I}_{HR})\|_F^2$$

where $\phi_l$ = feature map at VGG layer $l$.

**Why perceptual > MSE:** MSE-optimal solution is the conditional mean $E[I_{HR} | I_{LR}]$, which is blurry (averages over all possible HR images).

**Proof MSE-optimal is conditional mean:**

$\hat{I}^* = \arg\min_{\hat{I}} E[\|I - \hat{I}\|^2 | I_{LR}]$

$\frac{\partial}{\partial \hat{I}} E[\|I - \hat{I}\|^2] = -2E[I - \hat{I}] = 0 \implies \hat{I}^* = E[I | I_{LR}] \quad \blacksquare$

### Step 6: Frequency Domain Analysis of SR
SR must recover high frequencies lost during downsampling.

**Nyquist theorem:** Maximum recoverable frequency from LR: $f_{\max} = \frac{1}{2s}$ cycles/pixel.

SR must hallucinate frequencies in $(\frac{1}{2s}, \frac{1}{2})$ — this is the learned prior!

### Step 7: RL for Super-Resolution — Sequential Refinement
**State:** current SR estimate $\hat{I}_{HR}^{(t)}$ + LR input

**Action:** select enhancement operation (sharpen, detail add, texture synthesis)

**Reward:** $r_t = \Delta\text{PSNR} + \lambda_1 \Delta\text{SSIM} + \lambda_2 \Delta\text{LPIPS}^{-1}$

**Connection to RL:** The agent iteratively refines the SR output, learning when to add detail vs. when to stop (avoiding artifacts). This sequential approach outperforms single-pass methods on complex textures.

In [ ]:
!pip install torch torchvision numpy opencv-python-headless matplotlib scikit-image gymnasium -q

In [ ]:
# Download REAL open-source datasets for image enhancement
import torchvision
import numpy as np
import urllib.request
import os

# CIFAR-10 real photographs (our base images to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(1000)])
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded as ground truth")

# BSD68 denoising benchmark (68 real test images)
bsd_url = "https://raw.githubusercontent.com/clausmichele/CBSD68-dataset/master/CBSD68/original/"
os.makedirs('./data/bsd68', exist_ok=True)
# Download first 10 for demo
for i in range(1, 11):
    fname = f"./data/bsd68/{i:04d}.png"
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f"{bsd_url}{i:04d}.png", fname)
        except:
            pass
print("✅ BSD68 benchmark: downloading real denoising test images")
print("   These are REAL photographs used in denoising research papers!")

In [ ]:
import os
import random
import math
import collections

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3 — Synthetic Dataset & Upscaling Methods

## Deep Derivation: MAP Estimation and Regularization for Super-Resolution

### Step 1: Maximum A Posteriori (MAP) Framework

Given the degradation model $\mathbf{y} = \mathbf{DHx} + \mathbf{n}$, the MAP estimate:

$$\hat{\mathbf{x}}_{\text{MAP}} = \arg\max_{\mathbf{x}} P(\mathbf{x} | \mathbf{y}) = \arg\min_{\mathbf{x}} \underbrace{\frac{1}{2\sigma^2}\|\mathbf{y} - \mathbf{DHx}\|^2}_{\text{data fidelity}} + \underbrace{\lambda R(\mathbf{x})}_{\text{regularizer}}$$

**Step-by-step derivation:**

1. By Bayes: $P(\mathbf{x}|\mathbf{y}) \propto P(\mathbf{y}|\mathbf{x}) P(\mathbf{x})$
2. Gaussian noise: $P(\mathbf{y}|\mathbf{x}) = \mathcal{N}(\mathbf{DHx}, \sigma^2\mathbf{I})$
3. $-\log P(\mathbf{y}|\mathbf{x}) = \frac{1}{2\sigma^2}\|\mathbf{y} - \mathbf{DHx}\|^2 + \text{const}$
4. $-\log P(\mathbf{x}) = \lambda R(\mathbf{x})$ for the chosen prior

### Step 2: Closed-Form Solution for Gaussian Prior (Wiener Filter)

With $R(\mathbf{x}) = \|\mathbf{x}\|^2$ (Tikhonov regularization):

$$\hat{\mathbf{x}} = (\mathbf{H}^T\mathbf{D}^T\mathbf{DH} + \lambda\sigma^2\mathbf{I})^{-1} \mathbf{H}^T\mathbf{D}^T \mathbf{y}$$

In the Fourier domain (assuming circular boundary conditions):

$$\hat{X}(\omega) = \frac{\bar{H}(\omega)\bar{D}(\omega)}{|D(\omega)|^2|H(\omega)|^2 + \lambda\sigma^2} Y(\omega)$$

**Proof this is equivalent to the Wiener filter:**

The Wiener filter minimizes $E[\|\hat{X} - X\|^2]$ and has the form:

$$W(\omega) = \frac{S_{XX}(\omega) \bar{H}(\omega)}{|H(\omega)|^2 S_{XX}(\omega) + S_{NN}(\omega)}$$

Setting $\lambda = S_{NN}/S_{XX} = \sigma^2/\sigma_x^2$ gives the Tikhonov solution. $\blacksquare$

### Step 3: Sparsity Prior and $\ell_1$ Regularization

For natural images, the gradient distribution is **super-Gaussian** (heavy-tailed, peaked at zero). Using $R(\mathbf{x}) = \|\nabla \mathbf{x}\|_1$ (TV regularization):

$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathbf{y} - \mathbf{DHx}\|^2 + \lambda \|\nabla \mathbf{x}\|_1$$

**Why $\ell_1$ promotes sparsity (proof):**

The proximal operator of $\|\cdot\|_1$ is the soft-thresholding operator:

$$\text{prox}_{\lambda\|\cdot\|_1}(z) = \text{sign}(z) \max(|z| - \lambda, 0)$$

This sets small values exactly to zero (sparse), unlike $\ell_2$ which shrinks but never zeroes. For $\lambda > |z|$, the output is exactly 0. $\blacksquare$

### Step 4: ADMM Algorithm for SR

The Alternating Direction Method of Multipliers solves the constrained optimization:

$$\min_{\mathbf{x}, \mathbf{z}} \frac{1}{2}\|\mathbf{y} - \mathbf{DHx}\|^2 + \lambda\|\mathbf{z}\|_1 \quad \text{s.t. } \mathbf{z} = \nabla\mathbf{x}$$

**ADMM iterations:**

$$\mathbf{x}^{k+1} = (\mathbf{H}^T\mathbf{D}^T\mathbf{DH} + \rho\nabla^T\nabla)^{-1}(\mathbf{H}^T\mathbf{D}^T\mathbf{y} + \rho\nabla^T(\mathbf{z}^k - \mathbf{u}^k))$$

$$\mathbf{z}^{k+1} = S_{\lambda/\rho}(\nabla\mathbf{x}^{k+1} + \mathbf{u}^k) \quad \text{(soft thresholding)}$$

$$\mathbf{u}^{k+1} = \mathbf{u}^k + \nabla\mathbf{x}^{k+1} - \mathbf{z}^{k+1}$$

### Step 5: RL Agent as Patch-Level Prior Selector

Our RL agent selects different upscaling methods per patch, effectively choosing different priors:

| Method | Implicit Prior | Best For |
|--------|---------------|----------|
| Bicubic | Smooth (Gaussian) | Flat regions |
| Edge-guided | Edge-preserving (TV) | Boundaries |
| Learned CNN | Data-driven (learned) | Textures |

The agent learns $\pi(a | f_{\text{patch}})$ — which prior to apply based on local content. This is a **mixture of experts** approach to regularization, where the Q-function serves as the gating network. $\blacksquare$

In [ ]:
HR_SIZE = 64
SCALE = 2
LR_SIZE = HR_SIZE // SCALE
PATCH_SIZE = 8  # LR patch size
NUM_IMAGES = 60


def generate_hr_image(size=HR_SIZE) -> np.ndarray:
    """Generate a high-resolution synthetic image with sharp details."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    # Gradient background
    for c in range(3):
        base = random.uniform(0.1, 0.3)
        img[:, :, c] = np.linspace(base, base + 0.15, size).reshape(1, -1)

    # Sharp edges and shapes
    for _ in range(random.randint(4, 8)):
        colour = np.array([random.uniform(0.3, 1.0) for _ in range(3)], dtype=np.float32)
        x, y = random.randint(0, size-6), random.randint(0, size-6)
        w, h = random.randint(4, size//3), random.randint(4, size//3)
        img[y:y+h, x:x+w] = colour

    # Fine lines
    img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    for _ in range(random.randint(2, 4)):
        pt1 = (random.randint(0, size), random.randint(0, size))
        pt2 = (random.randint(0, size), random.randint(0, size))
        c = tuple(int(random.uniform(100, 255)) for _ in range(3))
        cv2.line(img_u8, pt1, pt2, c, 1)
    return np.clip(img_u8.astype(np.float32) / 255.0, 0, 1)


def create_lr(hr_img: np.ndarray, scale: int = SCALE) -> np.ndarray:
    """Downsample HR to create LR image."""
    blurred = cv2.GaussianBlur(hr_img, (3, 3), 0.8)
    lr = cv2.resize(blurred, (hr_img.shape[1] // scale, hr_img.shape[0] // scale),
                    interpolation=cv2.INTER_AREA)
    return lr.astype(np.float32)


dataset = []
for _ in range(NUM_IMAGES):
    hr = generate_hr_image()
    lr = create_lr(hr)
    dataset.append({"hr": hr, "lr": lr})

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(dataset[i]["hr"])
    axes[0, i].set_title(f"HR ({HR_SIZE}x{HR_SIZE})"); axes[0, i].axis("off")
    axes[1, i].imshow(dataset[i]["lr"])
    axes[1, i].set_title(f"LR ({LR_SIZE}x{LR_SIZE})"); axes[1, i].axis("off")
plt.suptitle("HR (top) vs LR (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## 4 — Upscaling Methods Implementation

In [ ]:
class SimpleSRCNN(nn.Module):
    """Lightweight SRCNN-style upscaler for 2x super-resolution."""

    def __init__(self, scale=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 3 * scale * scale, 3, padding=1),
            nn.PixelShuffle(scale),
        )

    def forward(self, x):
        return torch.clamp(self.net(x), 0, 1)


# Pre-train SRCNN briefly on our dataset
srcnn = SimpleSRCNN(SCALE).to(device)
srcnn_opt = optim.Adam(srcnn.parameters(), lr=1e-3)

print("Pre-training SRCNN...")
for epoch in range(80):
    total_loss = 0
    random.shuffle(dataset)
    for d in dataset:
        lr_t = torch.FloatTensor(d["lr"].transpose(2, 0, 1)).unsqueeze(0).to(device)
        hr_t = torch.FloatTensor(d["hr"].transpose(2, 0, 1)).unsqueeze(0).to(device)
        pred = srcnn(lr_t)
        loss = F.mse_loss(pred, hr_t)
        srcnn_opt.zero_grad()
        loss.backward()
        srcnn_opt.step()
        total_loss += loss.item()
    if epoch % 20 == 0:
        print(f"  Epoch {epoch}: MSE = {total_loss/len(dataset):.6f}")
srcnn.eval()
print("SRCNN pre-training complete!")

In [ ]:
def upscale_bicubic(patch: np.ndarray, scale: int = SCALE) -> np.ndarray:
    h, w = patch.shape[:2]
    return cv2.resize(patch, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)


def upscale_edge_guided(patch: np.ndarray, scale: int = SCALE) -> np.ndarray:
    """Edge-guided upscaling: sharpen after bicubic along detected edges."""
    upscaled = cv2.resize(patch, (patch.shape[1]*scale, patch.shape[0]*scale), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor((upscaled * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    # Unsharp mask along edges
    blurred = cv2.GaussianBlur(upscaled, (3, 3), 1.0)
    sharpened = np.clip(upscaled + 0.5 * (upscaled - blurred), 0, 1).astype(np.float32)
    edge_mask = (edges > 0).astype(np.float32)[:, :, np.newaxis]
    edge_mask = cv2.GaussianBlur(edge_mask, (3, 3), 0.5)[:, :, np.newaxis] if edge_mask.ndim == 2 else cv2.GaussianBlur(edge_mask, (3, 3), 0.5)
    result = upscaled * (1 - edge_mask) + sharpened * edge_mask
    return np.clip(result, 0, 1).astype(np.float32)


def upscale_learned(patch: np.ndarray, model: nn.Module = srcnn, scale: int = SCALE) -> np.ndarray:
    """Use pre-trained SRCNN for upscaling."""
    with torch.no_grad():
        inp = torch.FloatTensor(patch.transpose(2, 0, 1)).unsqueeze(0).to(device)
        out = model(inp).squeeze(0).cpu().numpy().transpose(1, 2, 0)
    return np.clip(out, 0, 1).astype(np.float32)


UPSCALE_METHODS = [
    ("bicubic",      upscale_bicubic),
    ("edge_guided",  upscale_edge_guided),
    ("learned_cnn",  upscale_learned),
]

print(f"Upscaling methods: {[name for name, _ in UPSCALE_METHODS]}")

## Deep Derivation: Perceptual Loss and Frequency Analysis of Super-Resolution

### Step 1: Perceptual Loss — Feature-Space Distance

Instead of pixel-wise MSE, compare images in a pretrained CNN's feature space:

$$\mathcal{L}_{\text{perceptual}} = \sum_{l \in \mathcal{L}} \frac{\lambda_l}{C_l H_l W_l} \|F^l(I_{\text{SR}}) - F^l(I_{\text{HR}})\|_F^2$$

where $F^l$ are features at layer $l$ of a VGG network.

### Step 2: Why MSE Produces Blurry Results — Formal Proof

**Theorem:** The MSE-optimal estimate is the conditional mean:

$$\hat{I}_{\text{MSE}}^* = \arg\min_{\hat{I}} E[\|I_{\text{HR}} - \hat{I}\|^2 | I_{\text{LR}}] = E[I_{\text{HR}} | I_{\text{LR}}]$$

**Proof:**

$$\frac{\partial}{\partial \hat{I}} E[\|I - \hat{I}\|^2 | I_{\text{LR}}] = -2E[I - \hat{I} | I_{\text{LR}}] = 0$$

$$\implies \hat{I}^* = E[I | I_{\text{LR}}]$$

**Why this is blurry:** When multiple HR images are consistent with a given LR image, the conditional mean **averages** them:

$$E[I_{\text{HR}} | I_{\text{LR}}] = \int I \cdot P(I | I_{\text{LR}}) dI$$

Averaging sharp images with different high-frequency details produces a blurry result (high-frequency components cancel out). $\blacksquare$

### Step 3: Perceptual Loss Avoids Blurriness

Perceptual loss operates in feature space where high-frequency details are represented as activation patterns. Matching features forces the network to produce **one plausible sharp image** rather than the average of many:

$$\hat{I}_{\text{perc}}^* = \arg\min_{\hat{I}} \|F(I_{\text{HR}}) - F(\hat{I})\|^2$$

This has a **unique minimizer** (up to feature-space equivalence) that preserves high-frequency texture, unlike the MSE's average. $\blacksquare$

### Step 4: Frequency Domain Analysis of Super-Resolution

**Sampling theorem:** Downsampling by $s$ preserves frequencies below $\frac{f_s}{2s}$ (Nyquist limit).

**SR must recover frequencies in $\left(\frac{f_s}{2s}, \frac{f_s}{2}\right)$** — these are **aliased** or **destroyed** during downsampling.

**Proof aliasing occurs above Nyquist:**

For signal $x(t)$ sampled at rate $f_s$: $X_s(f) = \sum_k X(f - kf_s)$. Components at $f > f_s/2$ fold back into $[0, f_s/2]$, corrupting low-frequency content.

After $s\times$ downsampling, the new sampling rate is $f_s/s$, so frequencies above $f_s/(2s)$ alias. $\blacksquare$

### Step 5: Pixel Shuffle — Sub-Pixel Convolution Proof

The PixelShuffle operation rearranges a tensor of shape $(C \cdot s^2, H, W)$ to $(C, sH, sW)$:

$$I_{\text{HR}}(c, s \cdot i + \delta_y, s \cdot j + \delta_x) = F(c \cdot s^2 + \delta_y \cdot s + \delta_x, i, j)$$

**Proof PixelShuffle is a bijection (lossless):**

The mapping $(c, \delta_y, \delta_x, i, j) \mapsto (c, si + \delta_y, sj + \delta_x)$ is injective because:
- Given $(c, p, q)$ in the output: $i = \lfloor p/s \rfloor$, $\delta_y = p \mod s$, $j = \lfloor q/s \rfloor$, $\delta_x = q \mod s$
- This inverse mapping is unique for each output position
- Input size $= C \cdot s^2 \cdot H \cdot W = C \cdot (sH) \cdot (sW) = $ output size

Therefore PixelShuffle is a bijection — no information is lost or duplicated. $\blacksquare$

### Step 6: RL Agent's Advantage — Content-Adaptive Upscaling

For each patch, the optimal upscaling method depends on local content:

$$a^* = \arg\max_{a \in \{0,1,2\}} \text{PSNR}(f_a(P_{\text{LR}}), P_{\text{HR}})$$

The RL agent approximates $a^*$ via $\pi(a | f_{\text{patch}})$, learning associations like:
- High edge energy → edge-guided (action 1)
- High texture energy → learned CNN (action 2)
- Low edge + low texture → bicubic suffices (action 0)

This **mixture of experts** strategy cannot be achieved by any single upscaling method.

## 5 — Patch Feature Extraction & SR Environment

In [ ]:
def extract_patch_features(patch: np.ndarray) -> np.ndarray:
    """Extract a compact feature vector describing the patch content."""
    gray = np.mean(patch, axis=2)
    mu = np.mean(gray)
    sigma = np.std(gray) + 1e-8

    # Edge energy
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_energy = np.mean(np.abs(gx) + np.abs(gy))

    # Texture energy (Laplacian variance)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    texture = np.var(lap)

    # Flatness
    val_range = np.max(gray) - np.min(gray) + 1e-8
    flatness = 1.0 - sigma / val_range

    # Per-channel stats
    ch_means = [np.mean(patch[:, :, c]) for c in range(3)]

    return np.array([mu, sigma, edge_energy, texture, flatness] + ch_means, dtype=np.float32)


class SuperResEnv(gym.Env):
    """Patch-level RL environment for super-resolution."""

    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, dataset, patch_size=PATCH_SIZE, scale=SCALE):
        super().__init__()
        self.dataset = dataset
        self.patch_size = patch_size
        self.scale = scale
        self.num_methods = len(UPSCALE_METHODS)

        feat_dim = 8  # length of feature vector
        self.observation_space = spaces.Box(-10, 10, shape=(feat_dim,), dtype=np.float32)
        self.action_space = spaces.Discrete(self.num_methods)

    def _get_patches(self):
        """Extract non-overlapping patches from LR and HR images."""
        ps = self.patch_size
        lr_patches, hr_patches = [], []
        h, w = self.lr_img.shape[:2]
        for y in range(0, h - ps + 1, ps):
            for x in range(0, w - ps + 1, ps):
                lr_patches.append(self.lr_img[y:y+ps, x:x+ps])
                s = self.scale
                hr_patches.append(self.hr_img[y*s:(y+ps)*s, x*s:(x+ps)*s])
        return lr_patches, hr_patches

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = random.randint(0, len(self.dataset) - 1)
        self.hr_img = self.dataset[idx]["hr"].copy()
        self.lr_img = self.dataset[idx]["lr"].copy()
        self.lr_patches, self.hr_patches = self._get_patches()
        self.patch_idx = 0
        self.sr_patches = []
        self.action_map = []
        self.total_reward = 0.0

        features = extract_patch_features(self.lr_patches[0])
        return features, {"patch": 0, "total_patches": len(self.lr_patches)}

    def step(self, action: int):
        _, upscale_fn = UPSCALE_METHODS[action]
        lr_patch = self.lr_patches[self.patch_idx]
        hr_patch = self.hr_patches[self.patch_idx]

        sr_patch = upscale_fn(lr_patch)

        # Ensure size matches
        target_h, target_w = hr_patch.shape[:2]
        if sr_patch.shape[:2] != (target_h, target_w):
            sr_patch = cv2.resize(sr_patch, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
        sr_patch = np.clip(sr_patch, 0, 1).astype(np.float32)

        self.sr_patches.append(sr_patch)
        self.action_map.append(action)

        reward = float(compute_psnr(hr_patch, sr_patch, data_range=1.0))
        self.total_reward += reward
        self.patch_idx += 1

        terminated = self.patch_idx >= len(self.lr_patches)

        if not terminated:
            next_features = extract_patch_features(self.lr_patches[self.patch_idx])
        else:
            next_features = np.zeros(self.observation_space.shape, dtype=np.float32)

        info = {"patch": self.patch_idx, "patch_psnr": reward}
        if terminated:
            sr_img = self._assemble_sr()
            info["image_psnr"] = float(compute_psnr(self.hr_img, sr_img, data_range=1.0))
            info["image_ssim"] = float(compute_ssim(self.hr_img, sr_img, data_range=1.0, channel_axis=2))

        return next_features, reward, terminated, False, info

    def _assemble_sr(self) -> np.ndarray:
        """Reassemble SR patches into full image."""
        ps = self.patch_size * self.scale
        h, w = self.hr_img.shape[:2]
        sr = np.zeros_like(self.hr_img)
        idx = 0
        for y in range(0, h - ps + 1, ps):
            for x in range(0, w - ps + 1, ps):
                if idx < len(self.sr_patches):
                    sr[y:y+ps, x:x+ps] = self.sr_patches[idx]
                    idx += 1
        return np.clip(sr, 0, 1).astype(np.float32)


env = SuperResEnv(dataset)
obs, info = env.reset()
print(f"Feature vector shape: {obs.shape}")
print(f"Actions: {env.action_space.n} ({[n for n, _ in UPSCALE_METHODS]})")
print(f"Patches per image: {info['total_patches']}")

## 6 — DQN Agent for Patch-Level Decisions

In [ ]:
class PatchQNetwork(nn.Module):
    """MLP Q-network for patch feature inputs."""

    def __init__(self, feat_dim: int, num_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)

    def push(self, *t):
        self.buf.append(t)

    def sample(self, bs):
        batch = random.sample(self.buf, bs)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buf)

In [ ]:
NUM_EPISODES = 500
BATCH_SIZE = 128
GAMMA = 0.95
LR = 5e-4
BUFFER_SIZE = 20_000
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 400
TARGET_UPDATE = 15

feat_dim = env.observation_space.shape[0]
policy_net = PatchQNetwork(feat_dim, env.action_space.n).to(device)
target_net = PatchQNetwork(feat_dim, env.action_space.n).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

ep_rewards, ep_psnrs, ep_ssims, losses = [], [], [], []
global_step = 0

for ep in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward = 0.0
    total_patches = info["total_patches"]

    for t in range(total_patches):
        eps = EPS_END + (EPS_START - EPS_END) * math.exp(-global_step / EPS_DECAY)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward
        global_step += 1

        if len(replay_buffer) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)
            q_vals = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_net(next_states).max(1)[0]
                target_q = rewards + GAMMA * next_q * (1 - dones)
            loss = F.smooth_l1_loss(q_vals, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        if done:
            break

    ep_rewards.append(ep_reward / total_patches)
    if "image_psnr" in info:
        ep_psnrs.append(info["image_psnr"])
        ep_ssims.append(info["image_ssim"])

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 100 == 0:
        print(f"Ep {ep:4d} | Avg Patch PSNR: {np.mean(ep_rewards[-50:]):6.2f} | Image PSNR: {np.mean(ep_psnrs[-50:]):6.2f} | ε: {eps:.3f}")

print("\nTraining complete!")

## 7 — Training Curves

## Deep Derivation: Image Quality Assessment and SR Evaluation Metrics

### Step 1: PSNR for Super-Resolution — Limitations

$$\text{PSNR} = 10\log_{10}\frac{255^2}{\text{MSE}} = 20\log_{10}\frac{255}{\text{RMSE}}$$

**PSNR fails as a perceptual metric** because:

1. **Shift invariance failure:** A 1-pixel shift gives $\text{PSNR} \approx 20$ dB even though the images look identical.

2. **Texture hallucination penalty:** A sharp, plausible texture may have high MSE vs. the true texture (different realization of the same statistics).

**Proof of shift-invariance failure:**

For image $I$ and its shift $I'(x,y) = I(x+1, y)$:

$$\text{MSE} = \frac{1}{N}\sum_{x,y}(I(x,y) - I(x+1,y))^2 = \sigma_{\nabla_x I}^2$$

For natural images, $\sigma_{\nabla_x I}^2 \approx 10$-$100$, giving PSNR $\approx 20$-$30$ dB — suggesting "poor quality" for a perceptually perfect shift. $\blacksquare$

### Step 2: SSIM for Super-Resolution

SSIM better captures perceptual quality by comparing structure independently of luminance and contrast:

$$\text{SSIM}(x, y) = l(x,y)^{\alpha} \cdot c(x,y)^{\beta} \cdot s(x,y)^{\gamma}$$

For SR: SSIM penalizes blurriness (reduces $c(x,y)$ — contrast) more than PSNR, which only cares about MSE.

### Step 3: LPIPS — Learned Perceptual Similarity

$$\text{LPIPS}(I_1, I_2) = \sum_l \frac{1}{H_l W_l}\sum_{h,w} \|w_l \odot (\hat{F}^l_{hw}(I_1) - \hat{F}^l_{hw}(I_2))\|_2^2$$

where $\hat{F}^l$ are channel-normalized features and $w_l$ are learned channel weights.

**Proof LPIPS is a valid metric:**

1. $\text{LPIPS}(I, I) = 0$ (identity)
2. $\text{LPIPS}(I_1, I_2) = \text{LPIPS}(I_2, I_1)$ (symmetry, since $\|\cdot\|_2$ is symmetric)
3. Triangle inequality holds because it's a weighted sum of $\ell_2$ norms (each satisfying triangle inequality)

Therefore LPIPS defines a metric on the space of images. $\blacksquare$

### Step 4: Natural Image Statistics — Why SR is Learnable

Natural images have predictable statistical properties:

**Power spectrum:** $S(f) \propto f^{-\alpha}$ with $\alpha \approx 2$ (1/f² law).

**Proof:** Edges in images create discontinuities. The Fourier transform of a step function decays as $1/f$. Since natural images have edges in 2D, the power spectral density decays as $1/f^2$.

**Implication for SR:** High frequencies ($f > f_{\text{Nyquist}}$) follow the same $1/f^2$ law — they can be predicted from low-frequency content because natural images are structured, not random. This is precisely what the learned CNN upscaler exploits.

### Step 5: Computational Trade-off Analysis

| Method | Complexity per patch | PSNR gain |
|--------|---------------------|-----------|
| Bicubic | $O(s^2 k^2)$ | Baseline |
| Edge-guided | $O(s^2 k^2 + E)$ | +0.5-1 dB |
| Learned CNN | $O(s^2 C^2 k^2 L)$ | +1-3 dB |

where $E$ = edge detection cost, $C$ = channels, $L$ = layers.

**RL agent's computational saving:** By using bicubic on flat patches (cheap) and CNN only on complex patches (expensive), the agent achieves near-CNN quality at reduced average cost:

$$\text{Cost}_{\text{RL}} = p_{\text{flat}} \cdot C_{\text{bicubic}} + p_{\text{edge}} \cdot C_{\text{edge}} + p_{\text{texture}} \cdot C_{\text{CNN}} < C_{\text{CNN}} \quad \blacksquare$$

In [ ]:
def moving_avg(d, w=30):
    return np.convolve(d, np.ones(w)/w, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ep_rewards, alpha=0.25, color="steelblue")
axes[0].plot(moving_avg(ep_rewards), color="navy", lw=2)
axes[0].set(xlabel="Episode", ylabel="Avg Patch PSNR", title="Mean Patch PSNR per Episode")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_psnrs, alpha=0.25, color="coral")
axes[1].plot(moving_avg(ep_psnrs), color="darkred", lw=2)
axes[1].set(xlabel="Episode", ylabel="Image PSNR (dB)", title="Full-Image PSNR")
axes[1].grid(True, alpha=0.3)

axes[2].plot(losses[:1000], alpha=0.25, color="mediumseagreen")
if len(losses) > 30:
    axes[2].plot(moving_avg(losses[:1000]), color="darkgreen", lw=2)
axes[2].set(xlabel="Step", ylabel="Loss", title="DQN Loss")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Super-Resolution Agent — Training Dynamics", fontsize=14)
plt.tight_layout()
plt.show()

## 8 — Evaluation with Strategy Maps

In [ ]:
def evaluate_sr_agent(env, policy_net, n=4):
    results = []
    policy_net.eval()
    for _ in range(n):
        obs, info = env.reset()
        total_patches = info["total_patches"]
        for t in range(total_patches):
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
            obs, _, done, _, info = env.step(action)
            if done:
                break
        sr_img = env._assemble_sr()
        bicubic_full = cv2.resize(env.lr_img, (HR_SIZE, HR_SIZE), interpolation=cv2.INTER_CUBIC)
        results.append({
            "lr": env.lr_img.copy(), "hr": env.hr_img.copy(),
            "sr": sr_img, "bicubic": bicubic_full,
            "action_map": env.action_map.copy(),
            "psnr_sr": info.get("image_psnr", 0),
            "psnr_bicubic": float(compute_psnr(env.hr_img, np.clip(bicubic_full, 0, 1).astype(np.float32), data_range=1.0)),
        })
    policy_net.train()
    return results

results = evaluate_sr_agent(env, policy_net)

# Visualise
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
row_labels = ["LR Input", "Bicubic", "Agent SR", "HR (Target)"]
for i, r in enumerate(results):
    axes[0, i].imshow(cv2.resize(r["lr"], (HR_SIZE, HR_SIZE), interpolation=cv2.INTER_NEAREST))
    axes[0, i].set_title("LR (nearest)"); axes[0, i].axis("off")

    axes[1, i].imshow(np.clip(r["bicubic"], 0, 1))
    axes[1, i].set_title(f"Bicubic\nPSNR={r['psnr_bicubic']:.1f}"); axes[1, i].axis("off")

    axes[2, i].imshow(np.clip(r["sr"], 0, 1))
    axes[2, i].set_title(f"Agent SR\nPSNR={r['psnr_sr']:.1f}"); axes[2, i].axis("off")

    axes[3, i].imshow(r["hr"])
    axes[3, i].set_title("Ground Truth"); axes[3, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11)
plt.suptitle("Super-Resolution: LR → Bicubic vs Agent vs GT", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Strategy map: which upscaler was selected for each patch
method_colours = {0: [0.3, 0.5, 0.9], 1: [0.9, 0.5, 0.2], 2: [0.2, 0.8, 0.3]}
method_names = [name for name, _ in UPSCALE_METHODS]

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
if len(results) == 1:
    axes = [axes]

for i, r in enumerate(results):
    grid_w = LR_SIZE // PATCH_SIZE
    grid_h = LR_SIZE // PATCH_SIZE
    strategy_img = np.zeros((grid_h, grid_w, 3))
    for idx, act in enumerate(r["action_map"]):
        y, x = divmod(idx, grid_w)
        if y < grid_h:
            strategy_img[y, x] = method_colours[act]
    axes[i].imshow(strategy_img, interpolation="nearest")
    axes[i].set_title(f"Image {i} Strategy Map")
    axes[i].axis("off")

legend_handles = [mpatches.Patch(color=method_colours[i], label=method_names[i]) for i in range(len(method_names))]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=11)
plt.suptitle("Patch-Level Upscaling Strategy Maps", fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

## 9 — Key Takeaways

| Insight | Detail |
|---------|--------|
| **Region adaptivity** | Different image regions benefit from different upscaling strategies |
| **Feature design** | Edge energy and texture variance are strong signals for method selection |
| **Learned vs classical** | The CNN upscaler excels on textured regions; bicubic suffices for flat areas |
| **Computational trade-off** | RL enables selective use of expensive methods only where they help |
| **Extensions** | Larger scale factors, overlapping patches with blending, continuous action spaces |

---

*Next notebook → Module 7.5: HDR Tone Mapping Agent with perceptual quality optimisation.*